# CSCI 677 HW2, Part 2: Gaussian Splatting

This notebook prepares the required 27-train/3-test split from the complete 30-camera exhaustive COLMAP reconstruction, trains once to 6000 iterations, and evaluates saved models at iterations 2000 and 6000. Run the cells in order with the `.venv-1` kernel.

In [ ]:
# Verify the repaired local CUDA environment.
import importlib.util
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

PROJECT_ROOT = Path(r"C:\Users\richard\Downloads\csci677-as02")
REPO_ROOT = PROJECT_ROOT / "gaussian-splatting"
ASSIGNMENT_ROOT = PROJECT_ROOT / "south-building-hw3-subsets-15-30"
FULL_IMAGES = ASSIGNMENT_ROOT / "images30"
RECON_ROOT = ASSIGNMENT_ROOT / "image30exh"
GS_DATASET = ASSIGNMENT_ROOT / "gs30_exh"
MODEL_ROOT = PROJECT_ROOT / "output" / "gs30_exh_6000"
HOLDOUT = ["P1180198.JPG", "P1180208.JPG", "P1180218.JPG"]
sys.path.insert(0, str(REPO_ROOT))

EXPECTED_PYTHON = (PROJECT_ROOT / ".venv-1" / "Scripts" / "python.exe").resolve()
print(f"Python: {sys.version.split()[0]} ({sys.executable})")
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, f"Select the .venv-1 kernel, not {sys.executable}"
assert torch.version.cuda is not None, "CPU-only PyTorch is active; select the .venv-1 kernel."
assert torch.cuda.is_available(), "PyTorch cannot access the NVIDIA GPU. Restart the kernel and retry."
try:
    import diff_gaussian_rasterization
    from simple_knn._C import distCUDA2
    import plyfile
except ImportError as error:
    raise RuntimeError("A compiled dependency is unavailable. Restart the kernel, then rerun this cell.") from error

print(f"PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
# Build the scene layout expected by the reference loader.
# Keep all 30 photographs because held-out views still need ground truth images; test.txt controls the split.
images_dir = GS_DATASET / "images"
sparse_dir = GS_DATASET / "sparse" / "0"
images_dir.mkdir(parents=True, exist_ok=True)
sparse_dir.mkdir(parents=True, exist_ok=True)

for image_path in sorted(FULL_IMAGES.glob("*.JPG")):
    shutil.copy2(image_path, images_dir / image_path.name)
for filename in ("cameras.bin", "images.bin", "points3D.bin"):
    shutil.copy2(RECON_ROOT / filename, sparse_dir / filename)
(sparse_dir / "test.txt").write_text("\n".join(HOLDOUT) + "\n", encoding="utf-8")

all_names = {p.name for p in images_dir.glob("*.JPG")}
assert len(all_names) == 30, f"Expected 30 images, found {len(all_names)}"
assert set(HOLDOUT) <= all_names
print(f"Scene ready: {GS_DATASET}")
print(f"Training views: {len(all_names - set(HOLDOUT))}; test views: {len(HOLDOUT)}")

In [ ]:
# Train exactly once to 6000 iterations and save both requested states.
train_command = [
    sys.executable, "train.py",
    "-s", str(GS_DATASET),
    "-m", str(MODEL_ROOT),
    "--eval",
    "--iterations", "6000",
    "--test_iterations", "2000", "6000",
    "--save_iterations", "2000", "6000",
    "--checkpoint_iterations", "2000", "6000",
]
started = time.perf_counter()
subprocess.run(train_command, cwd=REPO_ROOT, check=True)
training_seconds = time.perf_counter() - started
print(f"GS training time: {training_seconds:.1f} seconds ({training_seconds / 60:.2f} minutes)")

In [ ]:
# Render all training and test views from both requested saved iterations.
render_seconds = {}
for iteration in (2000, 6000):
    started = time.perf_counter()
    subprocess.run(
        [sys.executable, "render.py", "-m", str(MODEL_ROOT), "--iteration", str(iteration)],
        cwd=REPO_ROOT, check=True,
    )
    render_seconds[iteration] = time.perf_counter() - started
    print(f"Iteration {iteration} render time: {render_seconds[iteration]:.1f} seconds")

In [12]:
# Compute PSNR and SSIM for every train/test render directory.
# This avoids metrics.py's test-only aggregation and prints values needed in the report.
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor
from utils.image_utils import psnr
from utils.loss_utils import ssim

rows = []
for iteration in (2000, 6000):
    for split in ("train", "test"):
        result_dir = MODEL_ROOT / split / f"ours_{iteration}"
        render_dir, gt_dir = result_dir / "renders", result_dir / "gt"
        psnr_values, ssim_values = [], []
        for render_path in sorted(render_dir.glob("*.png")):
            gt_path = gt_dir / render_path.name
            render_tensor = (pil_to_tensor(Image.open(render_path).convert("RGB")).float().cuda() / 255.0).unsqueeze(0).contiguous()
            gt_tensor = (pil_to_tensor(Image.open(gt_path).convert("RGB")).float().cuda() / 255.0).unsqueeze(0).contiguous()
            psnr_values.append(psnr(render_tensor, gt_tensor).mean().item())
            ssim_values.append(ssim(render_tensor, gt_tensor).item())
        rows.append((iteration, split, len(psnr_values), sum(psnr_values)/len(psnr_values), sum(ssim_values)/len(ssim_values)))

print("iteration  split  views  PSNR(dB)  SSIM")
for iteration, split, count, mean_psnr, mean_ssim in rows:
    print(f"{iteration:9d}  {split:5s}  {count:5d}  {mean_psnr:8.3f}  {mean_ssim:.5f}")

iteration  split  views  PSNR(dB)  SSIM
     2000  train     27    23.410  0.68033
     2000  test       3    21.805  0.66005
     6000  train     27    26.689  0.82973
     6000  test       3    22.546  0.76605


In [13]:
# Save report-ready ground-truth/render comparisons for the three held-out views.
from PIL import ImageDraw

comparison_dir = PROJECT_ROOT / "output" / "comparisons"
comparison_dir.mkdir(parents=True, exist_ok=True)
for iteration in (2000, 6000):
    result_dir = MODEL_ROOT / "test" / f"ours_{iteration}"
    pairs = []
    for index, filename in enumerate(sorted(HOLDOUT)):
        gt = Image.open(result_dir / "gt" / f"{index:05d}.png").convert("RGB")
        rendered = Image.open(result_dir / "renders" / f"{index:05d}.png").convert("RGB")
        width = 700
        height = round(gt.height * width / gt.width)
        gt = gt.resize((width, height))
        rendered = rendered.resize((width, height))
        row = Image.new("RGB", (2 * width, height + 40), "white")
        row.paste(gt, (0, 40))
        row.paste(rendered, (width, 40))
        ImageDraw.Draw(row).text((10, 12), f"{filename}: ground truth", fill="black")
        ImageDraw.Draw(row).text((width + 10, 12), f"iteration {iteration}: render", fill="black")
        pairs.append(row)
    sheet = Image.new("RGB", (pairs[0].width, sum(row.height for row in pairs)), "white")
    y = 0
    for row in pairs:
        sheet.paste(row, (0, y))
        y += row.height
    output_path = comparison_dir / f"heldout_comparison_{iteration}.png"
    sheet.save(output_path)
    print(output_path)

C:\Users\richard\Downloads\csci677-as02\output\comparisons\heldout_comparison_2000.png
C:\Users\richard\Downloads\csci677-as02\output\comparisons\heldout_comparison_6000.png
